# Step 3: Simulation Study on Synthetic Data

This notebook implements Step 3 of the research plan: a systematic simulation study comparing four methods for PRS weight estimation on discrete (Binomial) genotype data.

**Methods compared:**
1. **Linear PRS** -- optimal linear weights from summary statistics
2. **Gaussian NN** -- summary-stat neural network under Gaussian genotype assumption (Part 1)
3. **Edgeworth NN** -- summary-stat neural network with cumulant corrections (Part 2)
4. **Oracle NN** -- individual-level neural network trained on raw genotype data (gold standard)

**Key question:** Does the Edgeworth correction for genotype non-Gaussianity improve PRS prediction, and does the improvement correlate with genotype skewness (as predicted by Theorem 1)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from ssnn.simulation import (
    SimulationScenario,
    run_scenario,
    run_single_rep,
)
from ssnn.cumulants import snp_cumulants

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

METHOD_COLORS = {
    "Linear PRS": "#4C72B0",
    "Gaussian NN": "#DD8452",
    "Edgeworth NN": "#55A868",
    "Oracle NN": "#C44E52",
}
METHOD_ORDER = ["Linear PRS", "Gaussian NN", "Edgeworth NN", "Oracle NN"]

## 1. Sanity Check: Single Scenario

Run one scenario with default parameters to verify the pipeline works end-to-end.

In [ ]:
sanity_scenario = SimulationScenario(
    p=30, m=3, n_train=3000, n_test=1000,
    maf_spectrum="mixed", ld_decay=0.5, heritability=0.5,
    sparsity=0.3, activation="relu",
)

sanity_results = run_scenario(sanity_scenario, n_reps=3, seed=42, verbose=True)

print("\n--- Sanity Check Results ---")
print(f"{'Method':<16s} {'Mean R2':>8s} {'Mean cos_sim':>12s}")
print("-" * 40)
for method in METHOD_ORDER:
    rows = [r for r in sanity_results if r["method"] == method]
    mean_r2 = np.mean([r["r2"] for r in rows])
    mean_cs = np.mean([r["weight_cosine"] for r in rows])
    print(f"{method:<16s} {mean_r2:8.4f} {mean_cs:12.4f}")

## 2. Main Comparison: Bar Chart of R-squared by Method

Default scenario with multiple replicates to show variance.

In [ ]:
default_scenario = SimulationScenario(
    p=30, m=3, n_train=3000, n_test=1000,
    maf_spectrum="mixed", ld_decay=0.5, heritability=0.5,
    sparsity=0.3, activation="relu",
)

default_results = run_scenario(default_scenario, n_reps=5, seed=100, verbose=True)

# Aggregate
r2_by_method = defaultdict(list)
for row in default_results:
    r2_by_method[row["method"]].append(row["r2"])

means = [np.mean(r2_by_method[m]) for m in METHOD_ORDER]
stds = [np.std(r2_by_method[m]) for m in METHOD_ORDER]
colors = [METHOD_COLORS[m] for m in METHOD_ORDER]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(METHOD_ORDER, means, yerr=stds, capsize=5, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Prediction $R^2$ (held-out test data)")
ax.set_title("Default Scenario: Mixed MAFs, p=30, h²=0.5")
ax.set_ylim(bottom=0)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{mean:.3f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## 3. Sensitivity: MAF Spectrum

How does prediction accuracy change across allele frequency spectra? This is the central question: does the Edgeworth correction help more when genotypes are highly skewed (rare variants)?

In [ ]:
maf_spectra = ["common", "mixed", "rare"]

maf_results = {}
for spec in maf_spectra:
    print(f"\n=== MAF spectrum: {spec} ===")
    scenario = SimulationScenario(
        p=30, m=3, n_train=3000, n_test=1000,
        maf_spectrum=spec, ld_decay=0.5, heritability=0.5,
        sparsity=0.3, activation="relu",
    )
    maf_results[spec] = run_scenario(scenario, n_reps=5, seed=200, verbose=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R^2 by MAF spectrum
ax = axes[0]
for method in METHOD_ORDER:
    means = []
    stds = []
    for spec in maf_spectra:
        vals = [r["r2"] for r in maf_results[spec] if r["method"] == method]
        means.append(np.mean(vals))
        stds.append(np.std(vals))
    ax.errorbar(maf_spectra, means, yerr=stds, marker="o", label=method,
                color=METHOD_COLORS[method], capsize=4, linewidth=2)
ax.set_xlabel("MAF Spectrum")
ax.set_ylabel("Prediction $R^2$")
ax.set_title("Prediction Accuracy vs. MAF Spectrum")
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)

# Edgeworth - Gaussian gap by MAF spectrum
ax = axes[1]
gaps = []
k3s = []
labels = []
for spec in maf_spectra:
    gauss_r2 = [r["r2"] for r in maf_results[spec] if r["method"] == "Gaussian NN"]
    ew_r2 = [r["r2"] for r in maf_results[spec] if r["method"] == "Edgeworth NN"]
    mean_gap = np.mean(ew_r2) - np.mean(gauss_r2)
    mean_k3 = np.mean([r["mean_abs_kappa3"] for r in maf_results[spec]])
    gaps.append(mean_gap)
    k3s.append(mean_k3)
    labels.append(spec)

ax.bar(labels, gaps, color="#55A868", edgecolor="black", linewidth=0.5)
for i, (lab, gap, k3) in enumerate(zip(labels, gaps, k3s)):
    ax.text(i, gap + 0.001 * np.sign(gap), f"|κ₃|={k3:.1f}", ha="center", fontsize=9)
ax.set_xlabel("MAF Spectrum")
ax.set_ylabel("$\\Delta R^2$ (Edgeworth − Gaussian)")
ax.set_title("Edgeworth Improvement by MAF Spectrum")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.show()

## 4. Sensitivity: LD Strength

Does the decorrelation approximation degrade at high LD? We sweep LD decay from 0 (identity) to 0.95 (very high LD).

In [ ]:
ld_decays = [0.0, 0.3, 0.5, 0.8, 0.95]

ld_results = {}
for decay in ld_decays:
    print(f"\n=== LD decay: {decay} ===")
    scenario = SimulationScenario(
        p=30, m=3, n_train=3000, n_test=1000,
        maf_spectrum="mixed", ld_decay=decay, heritability=0.5,
        sparsity=0.3, activation="relu",
    )
    ld_results[decay] = run_scenario(scenario, n_reps=5, seed=300, verbose=True)

fig, ax = plt.subplots(figsize=(9, 5))
for method in METHOD_ORDER:
    means = []
    stds = []
    for decay in ld_decays:
        vals = [r["r2"] for r in ld_results[decay] if r["method"] == method]
        means.append(np.mean(vals))
        stds.append(np.std(vals))
    ax.errorbar(ld_decays, means, yerr=stds, marker="o", label=method,
                color=METHOD_COLORS[method], capsize=4, linewidth=2)
ax.set_xlabel("LD Decay Parameter")
ax.set_ylabel("Prediction $R^2$")
ax.set_title("Prediction Accuracy vs. LD Strength")
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 5. Sensitivity: Heritability

Is the signal strong enough for nonlinearity to matter? We sweep h² from 0.1 to 0.8.

In [ ]:
h2_values = [0.1, 0.3, 0.5, 0.8]

h2_results = {}
for h2 in h2_values:
    print(f"\n=== h² = {h2} ===")
    scenario = SimulationScenario(
        p=30, m=3, n_train=3000, n_test=1000,
        maf_spectrum="mixed", ld_decay=0.5, heritability=h2,
        sparsity=0.3, activation="relu",
    )
    h2_results[h2] = run_scenario(scenario, n_reps=5, seed=400, verbose=True)

fig, ax = plt.subplots(figsize=(9, 5))
for method in METHOD_ORDER:
    means = []
    stds = []
    for h2 in h2_values:
        vals = [r["r2"] for r in h2_results[h2] if r["method"] == method]
        means.append(np.mean(vals))
        stds.append(np.std(vals))
    ax.errorbar(h2_values, means, yerr=stds, marker="o", label=method,
                color=METHOD_COLORS[method], capsize=4, linewidth=2)
ax.set_xlabel("Heritability ($h^2$)")
ax.set_ylabel("Prediction $R^2$")
ax.set_title("Prediction Accuracy vs. Heritability")
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 6. The Edgeworth Advantage

Scatter plot of (Edgeworth R² − Gaussian R²) vs. mean |κ₃| across all scenarios run so far. This tests the prediction from the Non-Gaussian Gap Theorem that the Edgeworth correction matters most when the standardized third cumulant is large.

In [ ]:
all_scenario_data = []

# Gather results from every sweep
sweep_registry = [
    ("Default", {"mixed": default_results}),
    ("MAF", maf_results),
    ("LD", ld_results),
    ("h²", h2_results),
]

for sweep_name, results_dict in sweep_registry:
    for param_val, rows in results_dict.items():
        gauss_r2 = [r["r2"] for r in rows if r["method"] == "Gaussian NN"]
        ew_r2 = [r["r2"] for r in rows if r["method"] == "Edgeworth NN"]
        oracle_r2 = [r["r2"] for r in rows if r["method"] == "Oracle NN"]
        linear_r2 = [r["r2"] for r in rows if r["method"] == "Linear PRS"]
        mean_k3 = np.mean([r["mean_abs_kappa3"] for r in rows])

        all_scenario_data.append({
            "sweep": sweep_name,
            "param": param_val,
            "mean_abs_kappa3": mean_k3,
            "delta_r2": np.mean(ew_r2) - np.mean(gauss_r2),
            "linear_r2": np.mean(linear_r2),
            "gauss_r2": np.mean(gauss_r2),
            "ew_r2": np.mean(ew_r2),
            "oracle_r2": np.mean(oracle_r2),
        })

fig, ax = plt.subplots(figsize=(8, 5))
for item in all_scenario_data:
    ax.scatter(
        item["mean_abs_kappa3"], item["delta_r2"],
        s=80, edgecolors="black", linewidth=0.5,
        color="#55A868", zorder=3,
    )
    ax.annotate(
        f"{item['sweep']}={item['param']}",
        (item["mean_abs_kappa3"], item["delta_r2"]),
        textcoords="offset points", xytext=(5, 5), fontsize=7,
    )

ax.set_xlabel("Mean Standardized |κ₃| (genotype skewness)")
ax.set_ylabel("$\\Delta R^2$ (Edgeworth − Gaussian)")
ax.set_title("Edgeworth Advantage vs. Genotype Skewness")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Summary Table

All scenarios, all methods, mean R² across replicates.

In [ ]:
header = f"{'Sweep':<8s} {'Param':<10s} {'|κ₃|':>6s} {'Linear':>8s} {'GaussNN':>8s} {'EdgewNN':>8s} {'Oracle':>8s} {'ΔR²':>8s}"
print(header)
print("-" * len(header))

for item in all_scenario_data:
    print(
        f"{item['sweep']:<8s} {str(item['param']):<10s} "
        f"{item['mean_abs_kappa3']:6.2f} "
        f"{item['linear_r2']:8.4f} "
        f"{item['gauss_r2']:8.4f} "
        f"{item['ew_r2']:8.4f} "
        f"{item['oracle_r2']:8.4f} "
        f"{item['delta_r2']:+8.4f}"
    )

print()
print("ΔR² = Edgeworth R² − Gaussian R² (positive = Edgeworth improves)")
print("|κ₃| = mean absolute standardized third cumulant (higher = more non-Gaussian)")